In [1]:
import os
import json
import uuid
from tqdm import tqdm
from dotenv import load_dotenv

from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings
from openai import OpenAI

/home/pablo/Documentos/ictl-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# =====================================
# Configuración entorno
# =====================================

load_dotenv()

client = OpenAI()

STRUCTURED_PATH = "/home/pablo/Documentos/ictl-rag/rag-asistent/data_procesor/structured_sections.json"

SEMANTIC_OUTPUT = "/home/pablo/Documentos/ictl-rag/rag-asistent/data_procesor/semantic_chunks.json"

EMBEDDING_OUTPUT = "/home/pablo/Documentos/ictl-rag/rag-asistent/data_procesor/chunks_with_embeddings.json"


In [3]:
with open(STRUCTURED_PATH, "r", encoding="utf-8") as f:
    sections = json.load(f)

print("Secciones cargadas:", len(sections))

Secciones cargadas: 465


In [4]:
def flatten_sections(sections):

    flat = []

    for sec in sections:

        for el in sec["elements"]:

            text = el.get("text", "")

            if not text:
                continue

            flat.append({
                **el,
                "section_title": sec["section_title"],
                "section_id": sec["section_id"],
                "page_start": sec["page_start"],
                "page_end": sec["page_end"]
            })

    return flat


flat_elements = flatten_sections(sections)

print("Elementos planos:", len(flat_elements))


Elementos planos: 516


In [5]:
langchain_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

chunker = SemanticChunker(
    langchain_embeddings
)


In [6]:
def semantic_chunk_elements(flat_elements):

    chunks = []

    for el in tqdm(flat_elements):

        text = el["text"]

        if len(text) < 40:
            continue

        semantic_chunks = chunker.split_text(text)

        for chunk_text in semantic_chunks:

            chunks.append({
                "chunk_id": str(uuid.uuid4()),
                "text": chunk_text,
                "page": el.get("page"),
                "source": el.get("source"),
                "section_title": el["section_title"],
                "section_id": el["section_id"],
                "parent_element_id": el["element_id"],
                "page_start": el["page_start"],
                "page_end": el["page_end"]
            })

    return chunks


In [7]:
semantic_chunks = semantic_chunk_elements(flat_elements)

print("Chunks generados:", len(semantic_chunks))


100%|██████████| 516/516 [00:47<00:00, 10.84it/s]

Chunks generados: 408


In [8]:
with open(SEMANTIC_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(semantic_chunks, f, ensure_ascii=False, indent=2)

print("Semantic chunks guardados")


Semantic chunks guardados
